In [ ]:
# Instalação de todas as dependências
!pip install scikit-learn pandas numpy optuna xgboost scipy shap matplotlib seaborn s3fs pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.0/102.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 57.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.7.0 which is incompatible.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.


In [ ]:
 # ======================================
# IMPORTANDO BIBLIOTECAS
# ======================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Visualização da árvore
from sklearn.tree import plot_tree

# Normalização
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer

#Métricas de classificação
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.preprocessing import label_binarize
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

#Feature selection
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest, f_classif, chi2, mutual_info_classif
from sklearn.feature_selection import RFECV

#Validação
from sklearn.model_selection import learning_curve
from sklearn.model_selection import validation_curve
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from scipy.stats import ks_2samp

#Validação cruzada
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, cross_val_score
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import optuna
from scipy.stats import uniform, randint
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint, loguniform

#SHAP
import shap

#Imputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer



# **Treino do modelo e métricas de validação**

In [ ]:
# ======================================
# CONECTANDO NA GOLD (BASE DE DADOS)
# ======================================

# VARIAVEIS
s3_path_comparativo_metas_resultados = "s3://fiap-datalake-tech-public/gold/comparativo_metas_resultados/"


#ACESSO
dados = pd.read_parquet(
    s3_path_comparativo_metas_resultados,
    engine="pyarrow",
    storage_options={"anon": True}
)

dados.head(n=5)


,ano,sigla_uf,id_municipio,nome_municipio,total_avaliados,taxa_alfabetizacao_real,meta_ano,desvio_meta,status_meta,_gold_processed_at,_analytics_version
0,2023,RO,1100072,Corumbiara,119,59.66,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3
1,2023,RO,1100122,Ji-Paraná,1496,70.52,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3
2,2023,RO,1101450,Parecis,55,56.36,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3
3,2023,AM,1300631,Beruri,243,89.71,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3
4,2023,AM,1301605,Fonte Boa,380,38.95,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3


In [ ]:
pib_renda = pd.read_csv('pib_renda_gold.csv')

# # Filtra apenas colunas necessárias e remove duplicatas de id_municipio
pib_renda_unico = pib_renda[[
    'id_municipio',
    'pib_per_capita_R$',
    'renda_media_per_capita_R$',
    'densidade_demografica'
]].drop_duplicates(subset=['id_municipio'])

# Merge sem gerar linhas duplicadas
dados_mais_pib_renda = dados.merge(
    pib_renda_unico,
    on='id_municipio',
    how='left'
)
dados_mais_pib_renda.head()

,ano,sigla_uf,id_municipio,nome_municipio,total_avaliados,taxa_alfabetizacao_real,meta_ano,desvio_meta,status_meta,_gold_processed_at,_analytics_version,pib_per_capita_R$,renda_media_per_capita_R$,densidade_demografica
0,2023,RO,1100072,Corumbiara,119,59.66,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,67914.967346,402.15,2.451961
1,2023,RO,1100122,Ji-Paraná,1496,70.52,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,47065.679747,743.35,17.942874
2,2023,RO,1101450,Parecis,55,56.36,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,49413.928396,343.73,1.599843
3,2023,AM,1300631,Beruri,243,89.71,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,12680.615355,193.40,1.183025
4,2023,AM,1301605,Fonte Boa,380,38.95,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,12604.111612,200.40,2.116989


In [ ]:
idhm = pd.read_csv('idhm_gold.csv')

# Merge sem gerar linhas duplicadas
dados_mais_pib_renda_idhm = dados_mais_pib_renda.merge(
    idhm[['id_municipio', 'idhm']],
    on='id_municipio',
    how='left'
)
dados_mais_pib_renda_idhm.head()

,ano,sigla_uf,id_municipio,nome_municipio,total_avaliados,taxa_alfabetizacao_real,meta_ano,desvio_meta,status_meta,_gold_processed_at,_analytics_version,pib_per_capita_R$,renda_media_per_capita_R$,densidade_demografica,idhm
0,2023,RO,1100072,Corumbiara,119,59.66,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,67914.967346,402.15,2.451961,0.613
1,2023,RO,1100122,Ji-Paraná,1496,70.52,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,47065.679747,743.35,17.942874,0.714
2,2023,RO,1101450,Parecis,55,56.36,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,49413.928396,343.73,1.599843,0.617
3,2023,AM,1300631,Beruri,243,89.71,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,12680.615355,193.40,1.183025,0.506
4,2023,AM,1301605,Fonte Boa,380,38.95,NaN,NaN,Sem Meta Definida,2026-07-14 21:33:01.491955,v3.0_spark_native_s3,12604.111612,200.40,2.116989,0.530


In [15]:
# ============================================================================================
# 1. DEFININDO X e y
#    Temporal:
#       2024 + 2025 -> TREINAMENTO
#       2026        -> TESTE FUTURO
#
#    IMPORTANTE:
#    Os dados de 2026 ainda não estão disponíveis.
#    Portanto, o conjunto de teste será criado somente em 2027,
#    quando os dados de 2026 estiverem disponíveis.
# ============================================================================================


# Ordenação temporal por município

dados_modelo = dados_mais_pib_renda_idhm.sort_values(
    by=["id_municipio", "ano"]
).reset_index(drop=True)


# ============================================================================================
# 2. FEATURE EXTRACTION
# ============================================================================================

status_desejados = [
    "Abaixo da Meta",
    "Atingiu a Meta"
]


# NÃO filtramos dados_modelo aqui pelo status_meta.
#
# O filtro será feito somente depois da separação temporal.
#
# Isso permite que, futuramente, os registros de 2026 sejam incorporados
# mesmo que ainda não possuam status_meta.


# ============================================================================================
# 3. FEATURE ENGINEERING TEMPORAL
# ============================================================================================

# Ordenação temporal

dados_modelo = dados_modelo.sort_values(
    by=["id_municipio", "ano"]
).reset_index(drop=True)


# --------------------------------------------------------------------------------------------
# Lag do ano anterior
# --------------------------------------------------------------------------------------------

dados_modelo["lag_1"] = (
    dados_modelo
    .groupby("id_municipio")["taxa_alfabetizacao_real"]
    .shift(1)
)


# --------------------------------------------------------------------------------------------
# Diferença em relação ao ano anterior
# --------------------------------------------------------------------------------------------

dados_modelo["diff_1"] = (
    dados_modelo
    .groupby("id_municipio")["taxa_alfabetizacao_real"]
    .diff(1)
)


# --------------------------------------------------------------------------------------------
# Média móvel dos 2 anos anteriores
# --------------------------------------------------------------------------------------------

dados_modelo["rolling_mean_2"] = (
    dados_modelo
    .groupby("id_municipio")["taxa_alfabetizacao_real"]
    .transform(
        lambda x: x.shift(1).rolling(
            2,
            min_periods=1
        ).mean()
    )
)


# ============================================================================================
# 4. SEPARAÇÃO TEMPORAL DO TREINAMENTO
# ============================================================================================

# 2024 + 2025 -> TREINAMENTO

dados_treino = dados_modelo[
    dados_modelo["ano"].isin([2024, 2025])
].copy()


# Manter somente as duas classes válidas do target

dados_treino = dados_treino[
    dados_treino["status_meta"].isin(status_desejados)
].copy()


print("Dados de treinamento:", dados_treino.shape)

print("\nDistribuição do target:")
print(
    dados_treino["status_meta"].value_counts()
)


# ============================================================================================
# 5. AGREGAÇÃO POR UF
# ============================================================================================

# Como 2024 e 2025 fazem parte do TREINAMENTO,
# podemos calcular a média estadual utilizando os dois anos.
#
# IMPORTANTE:
# Quando os dados de 2026 forem adicionados, eles NÃO deverão
# participar do cálculo dessa média antes da previsão.


media_alfa_uf_treino = (
    dados_modelo[
        dados_modelo["ano"].isin([2024, 2025])
    ]
    .groupby("sigla_uf")["taxa_alfabetizacao_real"]
    .mean()
    .rename("media_alfa_uf")
)


# Aplicar a média estadual ao conjunto de treinamento

dados_treino = dados_treino.merge(
    media_alfa_uf_treino,
    on="sigla_uf",
    how="left"
)


# ============================================================================================
# 6. DEFINIÇÃO DO X E y
# ============================================================================================

colunas_X = [
    "ano",
    "lag_1",
    "diff_1",
    "rolling_mean_2",
    "total_avaliados",
    "media_alfa_uf",
    "sigla_uf",
    "idhm",
    "renda_media_per_capita_R$",
    "pib_per_capita_R$",
    "densidade_demografica",
]


# X do treinamento

X_train = dados_treino[colunas_X].copy()


# y do treinamento

y_train = dados_treino["status_meta"].copy()


print("\nX_train:", X_train.shape)
print("y_train:", y_train.shape)


# ============================================================================================
# 7. REMOÇÃO DA COLUNA ANO
# ============================================================================================

X_train = X_train.drop(
    columns=["ano"]
).reset_index(drop=True)


y_train = y_train.reset_index(drop=True)


# ============================================================================================
# 8. FEATURE ENCODING — X
# ============================================================================================

# One-Hot Encoding do estado

X_train = pd.get_dummies(
    X_train,
    columns=["sigla_uf"],
    drop_first=True,
    dtype=int
)


# ============================================================================================
# 9. IMPUTER
# ============================================================================================

# O KNNImputer é ajustado SOMENTE no conjunto de treinamento.
#
# Em 2027, quando os dados de 2026 estiverem disponíveis,
# utilizaremos:
#
#     knn_imputer.transform(X_test)
#
# sem executar fit novamente.


knn_imputer = KNNImputer(
    n_neighbors=5,
    weights="uniform",
    keep_empty_features=True
)


# Guardar as colunas do treinamento

colunas_X = X_train.columns


# FIT somente no treinamento

X_train_imputed = knn_imputer.fit_transform(
    X_train
)


# Reconstruir DataFrame

X_train = pd.DataFrame(
    X_train_imputed,
    columns=colunas_X
)


# ============================================================================================
# 10. FEATURE ENCODING — y
# ============================================================================================

# O LabelEncoder é ajustado somente no treinamento.

le_y = LabelEncoder()


y_train_encoded = le_y.fit_transform(
    y_train
)


print("\nClasses do target:")
print(le_y.classes_)


# ============================================================================================
# 11. ESCALONAMENTO
# ============================================================================================

# O scaler também é ajustado somente no treinamento.
#
# Em 2027, os dados de 2026 receberão apenas:
#
#     scaler.transform(X_test)


scaler = StandardScaler()


X_train_scaled = scaler.fit_transform(
    X_train
)


X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)


# ============================================================================================
# 12. FEATURE SELECTION — SELECT KBEST
# ============================================================================================

# A seleção de features também é aprendida SOMENTE no treinamento.
#
# Em 2027, os dados de 2026 receberão apenas:
#
#     selector.transform(X_test)


selector = SelectKBest(
    mutual_info_classif,
    k=10
)


X_train_selected = selector.fit_transform(
    X_train_scaled,
    y_train_encoded
)


# ============================================================================================
# 13. FEATURES SELECIONADAS
# ============================================================================================

selected_names = X_train.columns[
    selector.get_support()
]


print(
    "\nFeatures selecionadas (Mutual Info):"
)

print(
    selected_names.tolist()
)


# ============================================================================================
# 14. RESULTADO FINAL DO PRÉ-PROCESSAMENTO
# ============================================================================================

print(
    "\nShape final do treinamento:",
    X_train_selected.shape
)

print(
    "Shape do target:",
    y_train_encoded.shape
)


# ============================================================================================
# 15. IMPORTANTE — TESTE 2026
# ============================================================================================

# NÃO existe X_test neste momento porque os dados de 2026
# ainda não estão disponíveis.
#
# Em 2027, quando os dados de 2026 forem incorporados,
# o processamento deverá seguir:
#
#     X_test = dados_2026[colunas_X].copy()
#
#     X_test = pd.get_dummies(...)
#
#     X_test = X_test.reindex(
#         columns=X_train.columns,
#         fill_value=0
#     )
#
#     X_test_imputed = knn_imputer.transform(X_test)
#
#     X_test_scaled = scaler.transform(X_test)
#
#     X_test_selected = selector.transform(X_test_scaled)
#
#     previsoes_2026 = modelo.predict(X_test_selected)
#
# E somente depois que o status_meta real de 2026 estiver disponível
# será possível calcular as métricas de avaliação.



Dados de treinamento: (10579, 18)

Distribuição do target:
status_meta
Atingiu a Meta    6744
Abaixo da Meta    3835
Name: count, dtype: int64

X_train: (10579, 11)
y_train: (10579,)

Classes do target:
['Abaixo da Meta' 'Atingiu a Meta']

Features selecionadas (Mutual Info):
['lag_1', 'diff_1', 'rolling_mean_2', 'media_alfa_uf', 'idhm', 'renda_media_per_capita_R$', 'pib_per_capita_R$', 'densidade_demografica', 'sigla_uf_MG', 'sigla_uf_RS']

Shape final do treinamento: (10579, 10)
Shape do target: (10579,)


In [16]:
# # =============================================================================
# # TREINAMENTO COM REGRESSÃO LOGÍSTICA (OTIMIZADO COM GRID SEARCH)
# # =============================================================================
# 1. Definir a estratégia de validação cruzada temporal
tscv = TimeSeriesSplit(n_splits=4)

# 2. Definir o grid de parâmetros
param_grid = {
    'C': [0.00001, 0.0001, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02],
    'penalty': ['l1'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [3000]
}

# 3. Instanciar o modelo
modelo = LogisticRegression(class_weight='balanced', random_state=42)

# 4. Instanciar o GridSearchCV
grid_search = GridSearchCV(
    estimator=modelo,
    param_grid=param_grid,
    cv=tscv, #Para série temporal
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

# 5. Treinar nos dados de treino
grid_search.fit(X_train_selected, y_train_encoded)

# 6. Exibir os melhores resultados
print(f"Melhor score (Validação Cruzada): {grid_search.best_score_:.4f}")
print("Melhores Hiperparâmetros:", grid_search.best_params_)

Fitting 4 folds for each of 16 candidates, totalling 64 fits
Melhor score (Validação Cruzada): 0.8386
Melhores Hiperparâmetros: {'C': 0.02, 'max_iter': 3000, 'penalty': 'l1', 'solver': 'saga'}


In [17]:
# ===============================================================
# MÉTRICAS DE AVALIAÇÃO — MODELO TREINADO EM 2024 + 2025
# AVALIAÇÃO REALIZADA EM 2027 COM OS RESULTADOS DE 2026
# ===============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    auc
)

# ===============================================================
# 1. MELHOR MODELO
# ===============================================================

melhor_modelo = grid_search.best_estimator_


# ===============================================================
# 2. PROBABILIDADES PARA 2026
# ===============================================================

y_prob_matrix = melhor_modelo.predict_proba(X_test_selected)

# Probabilidade da classe positiva
y_prob_pos = y_prob_matrix[:, 1]


# ===============================================================
# 3. THRESHOLD
# ===============================================================

threshold = 0.40

y_pred_encoded = (
    y_prob_pos >= threshold
).astype(int)


# ===============================================================
# 4. CONVERTER PREVISÕES PARA OS RÓTULOS ORIGINAIS
# ===============================================================

df_resultado_2026["status_meta_estimado"] = (
    le_y.inverse_transform(y_pred_encoded)
)


# ===============================================================
# 5. PROBABILIDADE DE CONFIANÇA
# ===============================================================

df_resultado_2026["probabilidade_confianca"] = (
    y_prob_matrix.max(axis=1)
)


# ===============================================================
# 6. AGORA SIM: COMPARAR COM O RESULTADO REAL DE 2026
# ===============================================================

df_resultado_2026["acertou"] = (
    df_resultado_2026["status_meta"]
    == df_resultado_2026["status_meta_estimado"]
)


# ===============================================================
# 7. VISUALIZAR RESULTADOS
# ===============================================================

print("=== COMPARAÇÃO REALIDADE VS ESTIMADO (2026) ===")

print(
    df_resultado_2026[
        [
            "nome_municipio",
            "sigla_uf",
            "status_meta",
            "status_meta_estimado",
            "probabilidade_confianca",
            "acertou"
        ]
    ].head(10)
)


# ===============================================================
# 8. ACURÁCIA GERAL
# ===============================================================

acuracia_2026 = (
    df_resultado_2026["acertou"].mean()
)

print(
    f"\nAcurácia geral do modelo para 2026: "
    f"{acuracia_2026:.2%}\n"
)


# ===============================================================
# 9. CODIFICAR O STATUS REAL DE 2026
# ===============================================================

y_test = df_resultado_2026["status_meta"]

y_test_encoded = le_y.transform(y_test)


# ===============================================================
# 10. MÉTRICAS DE CLASSIFICAÇÃO
# ===============================================================

accuracy = accuracy_score(
    y_test_encoded,
    y_pred_encoded
)

precision_macro = precision_score(
    y_test_encoded,
    y_pred_encoded,
    average="macro",
    zero_division=0
)

recall_macro = recall_score(
    y_test_encoded,
    y_pred_encoded,
    average="macro",
    zero_division=0
)

f1_macro = f1_score(
    y_test_encoded,
    y_pred_encoded,
    average="macro",
    zero_division=0
)

print(f"Acurácia Geral: {accuracy:.4f}")
print(f"Precisão Macro: {precision_macro:.4f}")
print(f"Recall Macro:   {recall_macro:.4f}")
print(f"F1-Score Macro: {f1_macro:.4f}\n")


# ===============================================================
# 11. RELATÓRIO COMPLETO POR CLASSE
# ===============================================================

print("--- Relatório Completo por Classe ---")

print(
    classification_report(
        y_test_encoded,
        y_pred_encoded,
        target_names=le_y.classes_,
        zero_division=0
    )
)


# ===============================================================
# 12. MATRIZ DE CONFUSÃO
# ===============================================================

matriz = confusion_matrix(
    y_test_encoded,
    y_pred_encoded
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=le_y.classes_
)

disp.plot(cmap=plt.cm.Blues)

plt.title("Matriz de Confusão - 2026")
plt.show()


# ===============================================================
# 13. CURVA ROC
# ===============================================================

auc_score_geral = roc_auc_score(
    y_test_encoded,
    y_prob_pos
)

fpr, tpr, thresholds_roc = roc_curve(
    y_test_encoded,
    y_prob_pos
)

roc_auc = auc(fpr, tpr)


plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    color="b",
    lw=2,
    label=f"Curva ROC (AUC = {roc_auc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    color="gray",
    linestyle="--"
)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])

plt.xlabel("Taxa de Falsos Positivos (FPR)")
plt.ylabel("Taxa de Verdadeiros Positivos (TPR)")

plt.title("Curva ROC - 2026")

plt.legend(loc="lower right")
plt.grid(True)

plt.show()


print(
    f"ROC AUC Score Geral: "
    f"{auc_score_geral:.4f}"
)

NameError: name 'X_test_selected' is not defined

In [ ]:
# ===============================================================
# SHAP PARA EXPLICABILIDADE
# ===============================================================

# 1. Nomes das colunas selecionadas pelo SelectKBest
selected_cols = X_train.columns[selector.get_support()].tolist()

# 2. DataFrames
X_train_scaled_df = pd.DataFrame(X_train_selected, columns=selected_cols, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_selected, columns=selected_cols, index=X_test.index)

# 3. DataFrame RAW
X_test_raw_df = X_test[selected_cols].copy()

# 4. Instanciar o Explainer e gerar os SHAP values
modelo_final = melhor_modelo
masker = shap.maskers.Independent(X_train_scaled_df, max_samples=X_train_scaled_df.shape[0])
explainer = shap.LinearExplainer(modelo_final, masker)
explanation = explainer(X_test_scaled_df)

shap_values = explanation.values
expected_val = explainer.expected_value

# ===============================================================
# 5. VISÃO GLOBAL (Summary Plot)
# ===============================================================
print("Summary Plot (Beeswarm) - Valores Originais")
shap.summary_plot(shap_values, X_test_raw_df)

print("Summary Plot (Bar)")
shap.summary_plot(shap_values, X_test_raw_df, plot_type="bar")

# ===============================================================
# 6. DEPENDENCE PLOTS (Grid com as Top Features)
# ===============================================================
print("Dependence plots - Valores Originais")

# Seleciona as Top 6 variáveis com maior impacto médio absoluto
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_features = pd.Series(mean_abs_shap, index=selected_cols).sort_values(ascending=False).head(6).index

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, feat in zip(axes.flatten(), top_features):
    shap.dependence_plot(
        feat,
        shap_values,
        X_test_raw_df,
        ax=ax,
        show=False
    )
    ax.set_title(f"Dependência: {feat}", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("dependence_grid_valores_reais.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()

# ===============================================================
# 7. WATERFALL PLOTS (Instâncias Individuais)
# ===============================================================
print("Waterfall plots - Valores Originais")

for idx in [0, 10, 50]:
    if idx < len(X_test_raw_df):
        exp_instance = shap.Explanation(
            values=shap_values[idx],
            base_values=expected_val,
            data=X_test_raw_df.iloc[idx].values,
            feature_names=selected_cols
        )

        plt.figure(figsize=(8, 6))
        shap.waterfall_plot(exp_instance, show=False)
        plt.title(f"Instância índice {idx}", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.savefig(f"waterfall_idx_{idx}.png", dpi=150, bbox_inches="tight")
        plt.show()
        plt.close()

# ===============================================================
# 8. DECISION PLOT
# ===============================================================
print("Decision plot")
n_samples = min(100, len(X_test_raw_df))

plt.figure(figsize=(10, 6))
shap.decision_plot(
    expected_val,
    shap_values[:n_samples],
    X_test_raw_df.iloc[:n_samples],
    show=False
)
plt.tight_layout()
plt.savefig("decision_plot.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()